In [ ]:
# to be used in colab
# !git clone https://github.com/Break-Through-Tech/Accenture-1S-contract-review-challenge
# %cd Accenture-1S-contract-review-challenge
# !git checkout branchname

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

from src.chunking import get_chunks, remove_partial

In [3]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    """Custom evaluation metric calculation for multi-label classification."""
    predictions, labels = eval_pred
    # Apply sigmoid to convert logits to probabilities
    probs = 1 / (1 + np.exp(-predictions))
    
    # Use a default threshold of 0.5 for intermediate tracking
    preds = (probs > 0.5).astype(float)
    
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    samples_f1 = f1_score(labels, preds, average='samples', zero_division=0)
    
    return {
        "eval_micro_f1": micro_f1,
        "eval_macro_f1": macro_f1,
        "eval_samples_f1": samples_f1
    }

In [ ]:
def optimize_thresholds(predictions, labels):
    """Finds the optimal decision threshold for each category to maximize Macro F1."""
    probs = 1 / (1 + np.exp(predictions))
    num_classes = labels.shape[1]
    best_thresholds = np.full(num_classes, 0.5)
    
    # Simple grid search per class to find the optimal classification threshold
    threshold_grid = np.linspace(0.01, 0.99, 100)
    
    for i in range(num_classes):
        best_f1 = 0.0
        best_t = 0.5
        for t in threshold_grid:
            preds = (probs[:, i] > t).astype(float)
            score = f1_score(labels[:, i], preds, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = t
        best_thresholds[i] = best_t
        
    # Evaluate overall performance with optimized thresholds
    opt_preds = np.zeros_like(probs)
    for i in range(num_classes):
        opt_preds[:, i] = (probs[:, i] > best_thresholds[i]).astype(float)
        
    print("--- Optimized Threshold Metrics ---")
    print(f"Micro F1:   {f1_score(labels, opt_preds, average='micro', zero_division=0):.5f}")
    print(f"Macro F1:   {f1_score(labels, opt_preds, average='macro', zero_division=0):.5f}")
    print(f"Samples F1: {f1_score(labels, opt_preds, average='samples', zero_division=0):.5f}")
    
    return best_thresholds

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "nlpaueb/bert-base-uncased-contracts"
labels = relevant_categories  # Keep the same order everywhere

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ContractDataset(Dataset):
    def __init__(self, chunks):
        # Partial chunks don't provide reliable positive/negative labels.
        chunks = remove_partial(chunks)

        texts = chunks["text"].fillna("").astype(str).tolist()
        self.encodings = tokenizer(texts, truncation=True, max_length=512)

        self.targets = []
        for categories in chunks["categories"]:
            categories = set(categories or [])
            categories.discard("None")
            self.targets.append(
                [float(label in categories) for label in labels]
            )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item

def collate_batch(features):
    targets = torch.tensor(
        [feature.pop("labels") for feature in features],
        dtype=torch.float,
    )
    batch = DataCollatorWithPadding(tokenizer)(features)
    batch["labels"] = targets
    return batch

project_root = Path.cwd().parent
train_chunks = get_chunks(project_root / "data/train.json", tokenizer, relevant_categories)
val_chunks = get_chunks(project_root / "data/val.json", tokenizer, relevant_categories)

train_dataset = ContractDataset(train_chunks)
val_dataset = ContractDataset(val_chunks)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label,
)
# Improved hyperparameters with standard, robust settings for Transformers
optimized_args = TrainingArguments(
    output_dir="./contract-category-model-opt",
    eval_strategy="epoch",         # Correct parameter for recent transformers versions
    save_strategy="epoch",
    learning_rate=3e-5,
    lr_scheduler_type="cosine", 
    warmup_steps=100,              # Uses standard ratio parameter safely
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,           
    weight_decay=0.05,            
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1", 
    greater_is_better=True,
    report_to="none",
)

optimized_trainer = Trainer(
    model=model,
    args=optimized_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_batch,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# Run the optimized training sequence
optimized_trainer.train()

# Get predictions to run the threshold optimization routine
eval_predictions = optimized_trainer.predict(val_dataset)
best_thresholds = optimize_thresholds(eval_predictions.predictions, eval_predictions.label_ids)

# Save optimized artifacts
optimized_trainer.save_model("./contract-category-model-opt")

KeyboardInterrupt: 